# 06 - Push the encoder past Table 1 on TextFooler and BERT-Attack

**Goal:** beat the paper's TMD numbers under the paper's own protocol (the authors' attack code, adaptive attacks, 1000 test sentences) on at least one column. The realistic targets are the two where enc_adv3 is already level:

| | Paper TMD | enc_adv3 (n=1000) | Needed for a clear win |
|---|---|---|---|
| TextFooler | 50.0 | 48.8 | above ~54.5 |
| BERT-Attack | 55.2 | 53.8 | above ~59.6 |

"Clear win" = above the paper's number by more than the combined 95% sampling noise, which is what `T.compare` reports as BEATS.

**Variants tried (all start from notebook 05's encoders, reused from Drive):**
1. **Noise around the encoder output** (`enc_noise`): z = E(t, c) + sigma * noise, one draw, no "closest to the input" selection, so the output stays near the cleaned point. Three noise levels. No retraining.
2. **Two more adversarial training rounds**, `enc_adv4` (TextFooler, BERT-Attack and PWWS attacks against enc_adv3 on training texts) and `enc_adv5` (TextFooler and BERT-Attack against enc_adv4), each keeping the **best epoch on validation data** instead of the last.
3. The newest encoder with the best noise level from 1.

**Selection protocol (decided before seeing any result):** every variant is compared on **validation** sentences only (200 per attack). The winner is the variant with the highest mean of TextFooler and BERT-Attack accuracy under attack, among variants whose clean validation accuracy is within 1 point of TMD's. Only that one variant is then attacked on the 1000 test sentences, once. Trying many variants on the test set and keeping the best would inflate the result.

**Honesty notes for the report:**
- Variant 1 adds randomness, which (like TMD's own randomness) can confuse the attacker. If it wins, the transfer test at the end shows how much of the gain is real cleaning.
- enc_adv4/5 are trained on BERT-Attack examples (from training texts), so BERT-Attack is no longer a held-out attack for them.

**Cost (L4, estimates):** about 1.5 h of attack generation for the two new rounds, about 1 h of validation screening, then about 1-1.5 h for the final test run. Everything is cached on Drive and resumes after a disconnect. Run notebooks 00 and 05 first.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00 and 05
N_VAL = 200                        # validation sentences per attack, for choosing a variant
N_TEST = 1000                      # the single final test run
SCREEN_ATTACKS = ('textfooler', 'bertattack')     # the two columns we target
SIGMAS = (0.25, 0.5, 1.0)          # noise levels around the encoder output
ENC_EPOCHS = 30

# Same settings as notebook 05 (so its encoders and pairs are reused from the cache)
W_DEN = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=0.0, w_cls=0.0)
W_ADV = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=1.0, w_cls=1.0)
ROUNDS_05 = [
    ('enc_adv',  'enc_den',  [('textfooler', 'vanilla', 1000, 1), ('textfooler', 'enc_den', 1000, 2)]),
    ('enc_adv2', 'enc_adv',  [('textfooler', 'enc_adv', 1000, 3)]),
    ('enc_adv3', 'enc_adv2', [('pwws', 'vanilla', 1000, 4), ('pwws', 'enc_adv2', 1500, 5)]),
]
NEW_ROUNDS = [  # (new encoder, initialised from, new adversarial pairs on TRAIN texts: (attack, against, n, seed))
    ('enc_adv4', 'enc_adv3', [('textfooler', 'enc_adv3', 1000, 6), ('bertattack', 'enc_adv3', 1000, 7),
                              ('pwws', 'enc_adv3', 1000, 8)]),
    ('enc_adv5', 'enc_adv4', [('textfooler', 'enc_adv4', 1000, 9), ('bertattack', 'enc_adv4', 1000, 10)]),
]

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)
head = lab.model.classifier
aug = T.SynonymAugmenter(f'{DIR}/TextDefender/counterfitted_neighbors.json', rate=0.3, seed=0)
E_aug, _ = lab.embeddings('train', 'base', augment_fn=aug, name='train_aug')

In [ ]:
encs = {}
def get_enc(name, **kw):
    """Load an encoder round from Drive, or train it (same code as notebook 05)."""
    path = lab._path(f'projenc_{name}_L{gan.latent_dim}C{gan.code_dim}.pt')
    enc = T.ProjEncoder(gan.latent_dim, gan.code_dim).to(T.DEV)
    if os.path.exists(path):
        enc.load_state_dict(torch.load(path)); print('loaded', name)
        return enc.eval()
    print('training', name)
    enc = T.train_proj_encoder(gan, E_tr, Y_tr, E_va, Y_va, head=head, epochs=ENC_EPOCHS, **kw)
    torch.save(enc.state_dict(), path)
    return enc

def victim_for(name):
    if name == 'vanilla':
        return T.Victim(lab, mode='none')
    return T.Victim(lab, gan=gan, enc=encs[name], mode='tmd', proj_mode='encoder')

# Notebook 05's encoders and training pairs: all loaded from the Drive cache
encs['enc_den'] = get_enc('enc_den', E_aug=E_aug, **W_DEN)
all_pairs = []
for name, init, new_pairs in ROUNDS_05:
    for attack, against, n, seed in new_pairs:
        all_pairs.append(T.adversarial_pairs(lab, victim_for(against), against, attack, n, seed))
    adv = tuple(torch.cat([p[j] for p in all_pairs]) for j in range(3))
    encs[name] = get_enc(name, E_aug=E_aug, adv=adv, init=encs[init], **W_ADV)
print(f'{sum(len(p[0]) for p in all_pairs)} training pairs from notebook 05')

In [ ]:
# Validation adversarial examples (attacks on VALIDATION texts against vanilla BERT), used only to pick the best
# training epoch of the new rounds. Never test texts.
val_pairs = [T.adversarial_pairs(lab, victim_for('vanilla'), 'vanilla', a, 500, s, split='val')
             for a, s in (('textfooler', 11), ('bertattack', 12))]
val_adv = (torch.cat([p[0] for p in val_pairs]), torch.cat([p[2] for p in val_pairs]))
print(len(val_adv[0]), 'validation adversarial examples for epoch selection')

In [ ]:
# Two more adversarial rounds, each keeping the best epoch on validation (clean + adversarial validation accuracy)
for name, init, new_pairs in NEW_ROUNDS:
    for attack, against, n, seed in new_pairs:
        all_pairs.append(T.adversarial_pairs(lab, victim_for(against), against, attack, n, seed))
    adv = tuple(torch.cat([p[j] for p in all_pairs]) for j in range(3))
    print(f'{name}: {len(adv[0])} adversarial training pairs')
    encs[name] = get_enc(name, E_aug=E_aug, adv=adv, init=encs[init], select='best', val_adv=val_adv, **W_ADV)

## Screening on validation sentences
Same 200 validation sentences for every variant. Nothing here touches the test set.

In [ ]:
def enc_victim(name, sigma=None):
    if sigma is None:
        return T.Victim(lab, gan=gan, enc=encs[name], mode='tmd', proj_mode='encoder')
    return T.Victim(lab, gan=gan, enc=encs[name], mode='tmd', proj_mode='enc_noise', sigma_scale=sigma)

variants = {'baseline': T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K),   # TMD, for reference
            'enc_adv3_m9': enc_victim('enc_adv3')}
variants.update({f'enc_adv3_noise{s}': enc_victim('enc_adv3', s) for s in SIGMAS})
variants.update({'enc_adv4_m9': enc_victim('enc_adv4'), 'enc_adv5_m9': enc_victim('enc_adv5')})
for tag, v in variants.items():
    T.evaluate(lab, v, tag, E_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val', need_log=True)

# the newest encoder with the best noise level found on enc_adv3
noise_tab = T.screen(lab, [f'enc_adv3_noise{s}' for s in SIGMAS], N_VAL)
best_sigma = float(noise_tab.loc[noise_tab['mean'].idxmax(), 'variant'].split('noise')[1])
tag = f'enc_adv5_noise{best_sigma}'
variants[tag] = enc_victim('enc_adv5', best_sigma)
T.evaluate(lab, variants[tag], tag, E_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val', need_log=True)

table = T.screen(lab, list(variants), N_VAL)
display(table)

In [ ]:
# Selection rule (fixed in advance): best mean of TextFooler and BERT-Attack on validation, among variants whose
# clean validation accuracy is within 1 point of TMD's.
base_clean = table.loc[table.variant == 'baseline', 'clean'].item()
cand = table[(table.variant != 'baseline') & (table.clean >= base_clean - 1.0)]
chosen = cand.loc[cand['mean'].idxmax(), 'variant']
print('chosen variant:', chosen)
display(cand.sort_values('mean', ascending=False))

## The one test run
The chosen variant on the 1000 test sentences (all three attacks), against the paper's Table 1, our TMD (paired McNemar test, same sentences) and the transfer test. Run this once.

In [ ]:
v = variants[chosen]
ALL = ('textfooler', 'bertattack', 'pwws')
T.evaluate(lab, v, chosen, E_te, Y_te, attacks=ALL, n=N_TEST, need_log=True)
display(T.compare(lab, chosen, baseline_tag='baseline', n=N_TEST))
display(pd.DataFrame([T.mcnemar(lab, 'baseline', chosen, a, N_TEST) for a in ALL]))
for attack in ('textfooler', 'pwws'):
    print('transfer from vanilla,', attack)
    display(T.transfer_eval(lab, {'baseline': variants['baseline'], 'enc_adv3_m9': variants['enc_adv3_m9'], chosen: v},
                            'vanilla', attack, N_TEST))